# 1. Ingest: подготовка enriched chunks

В этой тетрадке выполняется первый этап RAG pipeline: подготовка корпуса к индексации и создание индекса.

Исходные PDF-файлы уже были предварительно нормализованы в JSON: из них извлечены только темы исторических эссе, без инструкций, критериев оценивания и других нерелевантных частей, темы были обогащены метаданными. На этом шаге мы не работаем с PDF напрямую, а преобразуем проверенные JSON-файлы в финальные `EnrichedChunk`. 

Для перевода pdf в json использовался chat-gpt. System и user prompts лежат в папке `prompts`. Исходники пфд-файлов можно найти на официальном сайте олимпиады [сайт](https://olimpiada.ru/activity/84/tasks/2024?class=11&year=2024) или на моем гугл диске [сайт](https://drive.google.com/drive/folders/1sqr_aYKkd_CKNno-GI5VVVIhSjjTqcOP?usp=sharing). После автоматического извлечения JSON-файлы были просмотрены и при необходимости исправлены вручную. Поэтому в этой тетрадке они рассматриваются как подготовленный curated corpus.

Единица индексации — одна тема исторического эссе. Это естественный смысловой фрагмент: тема содержит цитату, автора, исторический период, ключевые слова, тип интерпретации и аннотацию. Повторюсь, все это было выделено из темы при помощи chat-gpt и преобразовано в удобный json формат. 

In [1]:
from hw_rag_mcp.settings import get_settings

settings = get_settings()
settings.print_summary()

✓ Конфигурация загружена
  Env file:             /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/.env
  Repo root:            /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp
  LLM:                  GigaChat-2
  Embeddings:           EmbeddingsGigaR
  Qdrant path:          /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/data/vectorstore/qdrant
  Qdrant collection:    history_essay_chunks
  Langfuse base URL:    https://cloud.langfuse.com
  GigaChat credentials: заданы
  Langfuse credentials: заданы


## 1. Пути к данным, общая аналитика

В `data/raw_json` лежат нормализованные JSON-файлы. Каждый файл соответствует одному PDF-документу олимпиады. Один документ - это задания регионального или заключительного этапа одного года - с 2015 по 2024, за исключением заключительных этапов 2019 (не проводилось из-за ковида) и 2020 (было в усеченном формате по той же причине). ПС тут год означает начало школьного учебного года, т е заключительный этап 2019 проходил весной 2020 года.

В `data/chunks` будут сохранены финальные chunks:

- `chunks.json` — удобен для просмотра человеком;
- `chunks.jsonl` — удобен для последующей загрузки в векторную базу.

In [ ]:
import pandas as pd

report_df = pd.read_csv(settings.raw_json_dir)
report_df.head()

,PDF,Тем,Статус
0,reg_2015.pdf,13,valid
1,final_2015.pdf,15,valid
2,reg_2016.pdf,13,valid
3,final_2016.pdf,18,valid
4,reg_2017.pdf,15,valid


In [4]:
print(f"{report_df.shape[0]} pdf files were converted to jsons, total {report_df['Тем'].sum()} topics")

17 pdf files were converted to jsons, total 255 topics


## 2. Создание чанков

`ChunkBuilder` отвечает только за orchestration: он читает JSON-файлы, валидирует их через Pydantic-модели, создаёт `EnrichedChunk` и сохраняет результат.

Формирование `embedding_text`, `display_text` и `metadata` находится внутри модели `EnrichedChunk`, потому что это логика самого chunk, а не внешнего билдера.

In [ ]:
from hw_rag_mcp.chunks.build_chunks import ChunkBuilder

builder = ChunkBuilder()

chunks = builder.build_from_dir(
    input_dir=settings.raw_json_dir,
    output_dir=settings.chunks_dir,
)

In [6]:
print(f"{len(chunks)} chunks") 

255 chunks


Чанки автоматически сохранены в json и jsonl файлы в папку `data/chunks`.

----

Каждый `EnrichedChunk` состоит из базовых полей и вычисляемых представлений:

- `embedding_text` — короткий содержательный текст, который будет отправляться в embedding model;
- `display_text` — человекочитаемое представление chunk для вывода в demo, eval и отчёте;
- `metadata` — payload для векторной базы, который позволит возвращать `document_id`, `chunk_id`, `source`, год, этап, автора цитаты и другие поля вместе с результатами поиска.

Мы специально не отправляем в embeddings служебные поля вроде `chunk_id`, `source`, `grades` и `document_id`, потому что они нужны для ссылок и фильтрации, а не для семантического поиска.

-----

Почему `embedding_text` и `display_text` не сохраняются как обычные поля

`embedding_text`, `display_text` и `metadata` являются вычисляемыми представлениями `EnrichedChunk`.

В `chunks.json` сохраняются только базовые поля chunk, чтобы не дублировать данные. При необходимости Python-модель сама строит:

- `embedding_text` для embeddings;
- `display_text` для вывода пользователю;
- `metadata` для payload в Qdrant.

## 3. Человекочитаемое представление chunk

`display_text` нужен для демонстрации результата пользователю, проверки в тетрадке и последующего отображения top-k результатов.

Он содержит служебные поля: источник, год, этап, номер темы и стабильный `chunk_id`.

In [7]:
# Берем первый chunk
first_chunk = chunks[0]

print(first_chunk.display_text)

Тема исторического эссе №1.
Документ: final_2015.
Источник: final_2015.pdf.
Год: 2015.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: В.В. Мавродин.
Исторический период: Киевская Русь конца IX-X веков.
Ключевые слова: Киевская держава; конец IX-X веков; восточное славянство; объединение русских земель; единый государственный организм; рубежи Руси; общность языка и культуры.
Тип интерпретации: positive_assessment.
Позиция автора: Автор положительно оценивает Киевскую державу как силу, объединявшую восточных славян и защищавшую рубежи Руси.
Аннотация: Тема посвящена роли Киевской державы в объединении восточного славянства в конце IX-X веков.

Текст темы:
«Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государственный организм, сплачивая и тем самым усиливая его, создавая условия для дальнейшего укрепления общности языка, быта, культуры, и обороняла 

## 4. Текст для embeddings

`embedding_text` — это версия chunk, которая будет отправляться в embedding model.

Она намеренно короче, чем `display_text`: здесь оставлена только содержательная информация, полезная для semantic retrieval:

- исторический период;
- автор цитаты;
- ключевые слова;
- позиция автора;
- исходный текст темы.

Служебные поля вроде `document_id`, `chunk_id`, `source`, `stage` и `grades` не добавляются в `embedding_text`; они будут храниться в metadata.

In [8]:
print(first_chunk.embedding_text)

Исторический период: Киевская Русь конца IX-X веков.
Автор цитаты: В.В. Мавродин.
Ключевые слова: Киевская держава; конец IX-X веков; восточное славянство; объединение русских земель; единый государственный организм; рубежи Руси; общность языка и культуры.
Позиция автора: Автор положительно оценивает Киевскую державу как силу, объединявшую восточных славян и защищавшую рубежи Руси.

Текст темы:
Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государственный организм, сплачивая и тем самым усиливая его, создавая условия для дальнейшего укрепления общности языка, быта, культуры, и обороняла рубежи земель русского народа, рубежи Руси


## 5. Metadata / payload

`metadata` будет сохранена в векторной базе вместе с embedding vector.

Эти поля нужны для воспроизводимости и объяснимости поиска: при выдаче top-k результатов мы должны показать `document_id`, `chunk_id`, `source` и другие атрибуты найденного фрагмента.

In [9]:
# Metadata первого chunk
first_chunk.metadata

{'point_id': 1,
 'document_id': 'final_2015',
 'chunk_id': 'final_2015_topic_001',
 'source': 'final_2015.pdf',
 'year': 2015,
 'stage': 'final',
 'grades': '10-11',
 'topic_number': 1,
 'quote_author': 'В.В. Мавродин',
 'historical_period': 'Киевская Русь конца IX-X веков',
 'keywords': ['Киевская держава',
  'конец IX-X веков',
  'восточное славянство',
  'объединение русских земель',
  'единый государственный организм',
  'рубежи Руси',
  'общность языка и культуры'],
 'interpretation_type': 'positive_assessment',
 'interpretation_position': 'Автор положительно оценивает Киевскую державу как силу, объединявшую восточных славян и защищавшую рубежи Руси.',
 'notes': 'Тема посвящена роли Киевской державы в объединении восточного славянства в конце IX-X веков.'}

### Технический `point_id` и человекочитаемый `chunk_id`

В Qdrant каждый объект должен иметь технический идентификатор `point_id` - integer или uuid. Для простоты мы используем последовательные integer ID: `1`, `2`, `3`, ...

При этом для отчёта, поиска и MCP-выдачи сохраняется человекочитаемый `chunk_id`, например `final_2015_topic_001`.

Так мы разделяем два типа идентификаторов:

- `point_id` — внутренний ID векторной базы;
- `chunk_id` — стабильная ссылка на тему эссе в корпусе.

## 6. Инициализация GigaChat embeddings

In [ ]:
from langchain_gigachat import GigaChatEmbeddings

embedding_model = GigaChatEmbeddings(
    credentials=settings.gigachat_credentials,
    scope=settings.gigachat_scope,
    model=settings.gigachat_embeddings_model,
    verify_ssl_certs=False,
)

print("✓ Модель эмбеддингов инициализирована")
print(f"  Model: {settings.gigachat_embeddings_model}")

✓ Модель эмбеддингов инициализирована
  Model: EmbeddingsGigaR


## 7. Создание локального Qdrant-индекса

На этом шаге мы превращаем `embedding_text` каждого chunk в векторное представление.

Для embeddings используется GigaChatEmbeddings. В embedding model передаётся не весь `display_text`, а только `embedding_text`: он содержит исторический период, автора цитаты, ключевые слова, позицию автора и исходный текст темы.

Служебные поля (`document_id`, `chunk_id`, `source`, `year`, `stage`) не эмбеддятся, а сохраняются как metadata в векторной базе.

Теперь загружаем chunks в локальную Qdrant collection.

Коллекция пересоздаётся при каждом запуске тетрадки. Это сделано намеренно: если меняется стратегия `embedding_text`, состав keywords или структура metadata, индекс не должен накапливать старые версии chunks.

In [ ]:
from hw_rag_mcp.vectorstore.qdrant_indexer import QdrantIndexer

indexer = QdrantIndexer(
    path=settings.qdrant_path,
    collection_name=settings.qdrant_collection,
    embedding_model=embedding_model,
)

indexer.index_chunks(
    chunks=chunks,
    batch_size=32,
)

print("✓ Qdrant индекс создан")
print(f"  Path: {settings.qdrant_path}")
print(f"  Collection: {settings.qdrant_collection}")
print(f"  Points count: {indexer.count()}")

Проверим параметры созданной коллекции: количество points, размерность вектора и тип distance metric.

In [12]:
collection_info = indexer.get_collection_info()
collection_info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=255, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=2560, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimize

In [13]:
assert indexer.count() == len(chunks)

print("✓ Количество points в Qdrant совпадает с количеством chunks")

indexer.close()
print("✓ Qdrant client closed")

✓ Количество points в Qdrant совпадает с количеством chunks
✓ Qdrant client closed


После завершения ingest явно закрываем Qdrant client. Локальный Qdrant блокирует директорию хранения, поэтому другой notebook не сможет открыть ту же базу, пока первый client держит lock.

## 8. Итог N1

В этой тетрадке выполнен полный ingest pipeline:

| Шаг | Результат |
|---|---|
| Загрузка JSON | Прочитаны нормализованные файлы из `data/raw_json` |
| Создание chunks | Каждая тема эссе преобразована в один `EnrichedChunk` |
| `embedding_text` | Сформирован короткий содержательный текст для embeddings |
| `display_text` | Сформирован человекочитаемый текст для вывода результатов |
| `metadata` | Сохранены `document_id`, `chunk_id`, `source`, `year`, `stage`, `quote_author`, `historical_period`, `keywords` |
| Embeddings | Для каждого chunk построен embedding через GigaChatEmbeddings |
| Vector DB | Chunks загружены в локальную коллекцию Qdrant |

На выходе N1 мы получили готовый локальный индекс. Следующая тетрадка `n2_search_demo.ipynb` будет проверять качество поиска: dense search, keyword/BM25 baseline и top-k результаты.